# Lab 13: Generative Models

In this lab, we will have a look at two kinds of generative modeling frameworks: **variational autoencoders** (VAE) and **flow matching**. The VAE is a bit over 10 years old, making it "archaic" by today's standards. Flow matching, on the other hand, is much more recent and is behind the latest image and video generators, such as Google's Veo 3. It can be seen as a successor to diffusion models, which was the technology behind the first "boom" of realistic image generators, such as StableDiffusion. With the VAE, we will generate some MNIST samples, while we discuss how to arrange random 2d points into two "moons" with flow matching.

## Part 1: Variational Autoencoder (VAE)
The VAE [[1]](https://arxiv.org/abs/1312.6114) concerns itself with a particular kind of graphical model that consists of two variables connected by a single edge $\mathbf z \to \mathbf x$. Here, $\mathbf z$ is a _latent_ variable, while $\mathbf x$ is the _observed_ variable.

We are given a dataset $\mathbf X = \{\mathbf x^{(i)}\}_{i=1}^N$ that we assume to be generated by such a process. The process first draws a latent variable from a prior distribution $p_{\theta^*}(\mathbf z)$ and then samples the observed variable conditioned on the latent, using $p_{\theta^*}(\mathbf x \mid \mathbf z)$. Our aim is to sample from the data distribution $p_{\theta^*}(\mathbf x)$.
Even with access to $p_{\theta}(\mathbf z)$ and $p_{\theta}(\mathbf x \mid \mathbf z)$ for any $\theta$, we don't know the true parameters $\theta^*$, nor do we have access to the latent vectors $\mathbf z^{(i)}$ corresponding to the data points.

#### Question 1
Can the EM algorithm be used to solve this problem?

<hr>

Since the true posterior for given parameters $\theta$ is generally intractable, we use *variational inference*, effectively approximating the true posterior $p_{\theta^*}(\mathbf z \mid \mathbf x)$ by $q_{\phi}(\mathbf z \mid \mathbf x)$. If $q$ is defined through a neural network, $\phi$ represents its weights and biases, which are referred to as the *variational parameters* in this context.

In a typical VAE, the following assumptions are made:
* The prior distribution over $\mathbf z$ is a standard (multivariate) normal: $p_{\theta}(\mathbf z) = \mathcal N(\mathbf z; \mathbf 0, \mathbf I)$.
* The variational posterior also follows a (multivariate) normal distribution $q_\phi(\mathbf z \mid \mathbf x) = \mathcal N(\mathbf z; \boldsymbol\mu, \mathsf{diag}(\boldsymbol\sigma^2))$ where $\boldsymbol\mu = \boldsymbol\mu_\phi(\mathbf x)$ and $\log \boldsymbol\sigma^2 = \log \boldsymbol\sigma_\phi(\mathbf x)^2$ are computed by a neural network. This component is called the **encoder**.
* Finally, $p_\theta(\mathbf x \mid \mathbf z)$ is assumed to be a Gaussian in the case of real data, or a Bernoulli distribution in the case of binary data. In either case, the distribution parameters are also computed by a neural network. This component is called the **decoder**.

### Training the VAE
For training, we want to maximize the log-likelihood of the data, as usual. Recall from the lecture that this is bounded from below by what we called the *ELBO*:
\begin{align*}
    \log p_\theta(\mathbf x) = \underbrace{\mathbb E_{q_\phi(\mathbf z \mid \mathbf x)}\left[\log \frac{p_\theta(\mathbf x, \mathbf z)}{q_\phi(\mathbf z \mid \mathbf x)}\right]}_{=: \mathcal L(\theta, \phi;\mathbf x)} + D_\mathrm{KL}(q_\phi(\mathbf z \mid \mathbf x) \Vert p_\theta(\mathbf z \mid \mathbf x)) \geq \mathcal L(\theta, \phi; \mathbf x).
\end{align*}
Before proceeding, we bring the ELBO into a different form.

#### Question 2 (*)
Show that $\mathcal L(\theta, \phi; \mathbf x) = \mathbb E_{q_\phi(\mathbf z \mid \mathbf x)}[\log p_\theta(\mathbf x \mid \mathbf z)] - D_\mathrm{KL}(q_\phi(\mathbf z \mid \mathbf x) \Vert p_\theta(\mathbf z))$. How can each term be interpreted?

<hr>

To optimize the generative and variational parameters jointly, we use a variant of gradient descent on $\mathcal L$, such as Adam. If the prior and the encoder are Gaussians (as in our case), the second term can be differentiated analytically. For the first term, an estimate of the gradient is still necessary. To this end, the expectation is approximated via sampling, $\frac{1}{L} \sum_{\ell=1}^L \log p_\theta(\mathbf x \mid \mathbf z^{(\ell)})$, where $\mathbf z^{(\ell)} \sim q_\phi(\cdot \mid \mathbf x)$.

#### Question 3 (*)
Compute the gradient of the sampled expectation with respect to the generative parameters $\theta$. Why does the same approach not work for the variational parameters $\phi$? Propose a solution.

### Implementation
The following code is an implementation of a VAE. Make sure that you have `pytorch` and `torchvision` installed. Since we want to train on MNIST, which has pixel values in the interval $[0,1]$, we pick for the decoder a Bernoulli distribution.

In [ ]:
import torch
import torch.nn as nn
from torch.distributions.multivariate_normal import MultivariateNormal
from torchvision import datasets, transforms
from tqdm import tqdm

class Encoder(nn.Module):
    def __init__(self, n_latent = 10):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        self.l_mu = nn.Linear(256, n_latent)
        self.l_var = nn.Linear(256, n_latent)

    def forward(self, x):
        x = self.fc(x)
        mu = self.l_mu(x)
        log_var = self.l_var(x)
        return mu, log_var

class Decoder(nn.Module):
    def __init__(self, n_latent = 10):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(n_latent, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU()
        )
        self.l_probs = nn.Linear(512, 784)
        

    def forward(self, z):
        z = self.fc(z)
        probs = torch.sigmoid(self.l_probs(z))
        return probs

def initialize_weights(model):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)

batch_size = 128
n_data = 784
n_latent = 20
encoder = Encoder(n_latent)
decoder = Decoder(n_latent)

initialize_weights(encoder)
initialize_weights(decoder)

#### Question 4 (*)
Implement the loss function $-\mathcal L(\theta, \phi, \mathbf x)$ with $L = 1$ Monte Carlo samples for the expectation. For the KL divergence term, you may use the fact that $$D_\mathrm{KL}(\mathcal N(\;\cdot\;; \boldsymbol \mu, \mathsf{diag}(\boldsymbol\sigma^2)) \Vert \mathcal N(\;\cdot\;; \mathbf 0, \mathbf I)) = \frac{1}{2} \sum_{j=1}^J (1 + \log((\sigma_j)^2) - (\mu_j)^2 - (\sigma_j)^2),$$
where $J$ is the latent dimension.

In [ ]:
def loss_fn(x, x_rec, mu_z, log_var_z):
    pass

### Training on MNIST
We will train the VAE on the MNIST handwritten digits dataset. You do not have to add any code, but make sure to have a look at the components going into the loss function. 

In [ ]:
train_dataset = datasets.MNIST(root='./mnist_data/', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='./mnist_data/', train=False, transform=transforms.ToTensor(), download=False)

train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [ ]:
def train(n_epochs=100):
    for epoch in tqdm(range(n_epochs)):
        train_loss = 0
        for batch_idx, (x, _) in enumerate(train_loader):
            # Flatten the image
            x = x.view(-1, n_data)

            # Reset gradients
            optimizer.zero_grad()

            # Encode and use reparameterization trick
            mu_z, log_var_z = encoder(x)
            std_z = torch.exp(0.5 * log_var_z)
            eps = torch.randn_like(std_z)
            z = mu_z + eps * std_z

            # Reconstruct
            x_rec = decoder(z)

            # Compute loss
            loss = loss_fn(x, x_rec, mu_z, log_var_z)

            # Backpropagation and weight update
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        print(f"Epoch {epoch+1}: Train loss: {train_loss / len(train_loader.dataset)}")
        scheduler.step(train_loss)

train(10)

#### Question 5
Write a function `sample` that samples a batch of images from the VAE.

In [ ]:
import matplotlib.pyplot as plt

def sample(n_samples=16):
    pass

s = sample()
fig, ax = plt.subplots(4, 4)
for r in range(4):
    for c in range(4):
        ax[r][c].axis('off')
        ax[r][c].imshow(1-s[r*4+c], cmap='Greys')

### Conditional Sampling
Most of the time, we are not interested in generating arbitrary images from a dataset, but rather ones that belong to a certain *class* or, even more generally, match a certain description. This is called conditional or guided sampling. Assuming that images belonging to the same class lie close together in the latent space, all it takes it to find the regions in the latent space that belong to the different classes.

#### Question 6 (optional)
Pick a number $c \in \{0,\dots,9\}$, take a batch $\{\mathbf x^{(i)}\}_{i=1}^B$ of test images with label $c$ and compute the mean encoding vector $\boldsymbol\mu_c = \frac{1}{B} \sum_{i=1}^B \boldsymbol\mu_\phi(\mathbf x^{(i)})$. Then, generate samples by sampling latent vectors from $\mathcal N(\boldsymbol\mu_c, \mathbf I)$ instead of the standard normal distribution.

While this is a quick method, it only works if same-class data points actually form clusters in the latent space. Some other options include:
* Conditional VAE: Training the encoder and decoder with $[\mathbf x, \mathbf c]$ resp. $[\mathbf z, \mathbf c]$, where $\mathbf c$ is the one-hot encoded class vector and $[\cdot,\cdot]$ denotes vector concatenation.
* Classifier-Guided Sampling: For this, you can re-use the trained VAE, but you need to train an MNIST classifier $f = (f_0,\dots,f_9) \colon [0,1]^{28\times 28} \to [0,1]^{10}$. Then, use SGD to adjust $\mathbf z$ to maximize $f_c(\mathbf x)$, where $\mathbf x \sim p_\theta(\mathbf x \mid \mathbf z)$.
* Circling back to the method proposed in Question 6, we could attempt to improve it by modifying the loss function to enforce clustering behavior in the latent space: Instead of modeling $p(\mathbf z)$ as a standard normal distribution, we could try choosing a GMM.

## Part 2: Flow Matching

Flow matching tries to solve the same problem as VAEs: Given a dataset $\{x^{(i)}\}_{i=1}^N$ of observations from some unknown distribution $q(x)$, generate samples from that distribution.

The key idea behind flow matching is to move samples from an easy distribution (such as a Gaussian) to our target distribution. This can be achieved by training a deterministic, time-dependent velocity field $u_t(x)$, $t \in [0,1]$, which pushes each sample along a certain path.

The following task will help you understand this key idea a bit more.

#### Question 7
Consider two distributions over $\mathbb R^2$, namely $p = \mathcal N(\mathbf 0, \mathbf I)$ and $q = 0.5 \mathcal N((-1, 0)^\top, \mathbf I) + 0.5 \mathcal N((1, 0)^\top, \mathbf I)$. Sketch a velocity field $u_t$ that transports samples from $p$ in such a way that the resulting points look as if they had been sampled from $q$. Sketch it at three time points: $t = 0.1$, $t = 0.5$ and $t = 0.9$.

*Note: This is meant to be an intuitive task, so there should be no math involved.*

<hr>

### Flows
Every velocity field defines a **flow**. This flow is denoted by $\psi \colon [0,1] \times \mathbb R^d \to \mathbb R^d$. We typically use the shorthand $\psi_t(x) \coloneqq \psi(t, x)$.
The flow is implicitly defined through an _ordinary differential equation_ (ODE)
$$
    \frac{\mathrm d}{\mathrm d t} \psi_t(x) = u_t(\psi_t(x))
$$
with initial condition $\psi_0(x) = x$.

Furthermore, we say that the velocity field $u_t$ _generates_ the probability path $p_t$, if its flow $\psi_t$ satisfies $$X_t \coloneqq \psi_t(X_0) \sim p_t \quad \text{for} \quad X_0 \sim p_0.$$

In summary, the velocity field _defines_ the motion. The flow _is_ the motion. Hence, to generate samples from our target distribution, we need to
1. train a suitable velocity field $u_t$,
2. solve the ODE to get the flow $\psi_1$,
3. sample $z \sim p_0$ (easy) and apply the flow $x = \psi_1(z)$.

The following figure from [2] illustrates this.

<div>
<img src="flow_matching.png" width="1000"/>
</div>

### Conditional Probability Paths and the Training Objective
We have reduced the sampling problem to the problem of finding a set of parameters $\theta$ to describe an appropriate velocity field $u_t^\theta(x)$. As usual, we assume the source distribution to be Gaussian, i.e., $p = p_0 = \mathcal N(0, I)$.

Next, to define the remaining distributions of our probability path $p_t$ for $t \in (0,1)$, we do the simplest thing imaginable: _linear interpolation_. In other words, we define $$X_t \coloneqq t X_1 + (1-t) X_0,$$ where $X_0 \sim p_0$ and $X_1 \sim p_1$. The distribution of $X_t$ is what we would like $p_t$ to be. See subfigure (b) in the plot above.

In practice, we cannot sample $X_1$ (that is our goal, after all), so we have to resort to the existing samples that we already have. This results in the analogous definition
$$X_{t \mid 1} \coloneqq t x_1 + (1-t) X_0,$$
where $x_1$ is a sample from our dataset.

#### Question 8
Knowing that $X_0 \sim \mathcal N(0, I)$, express the distribution of $X_{t \mid 1}$, i.e., $p_{t \mid 1}(\cdot \mid x_1)$.

<hr>

Mathematically, we can express $p_t$ as a marginalization involving the conditional probability paths,
$$p_t(x) = \int p_{t \mid 1}(x \mid x_1) p_1(x_1) \mathrm d x_1.$$

The figure below shows an example of a conditional probability path as well as the marginal probability path.

<div>
<img src="probability_paths.png" width="500"/>
</div>

#### Question 9 (*)
Solve $\frac{\mathrm d}{\mathrm d t} X_{t \mid 1} = u_t(X_{t \mid 1} \mid x_1)$. Proceed as follows:

a) Use the definition of $X_{t \mid 1}$ to compute the derivative.<br>
b) Express $X_0$ in terms of $X_{t \mid 1}$ and $x_1$.<br>
c) Combine to get an expression for $u_t(X_{t \mid 1} \mid x_1)$.

<hr>

The parameters $\theta$ of the neural network defining the velocity field can be trained via regression. The objective is given by the _conditional flow matching loss_
$$ \mathcal L_\mathrm{CFM}(\theta) = \mathbb E_{t, X_t, X_1} \lVert u_t^\theta(X_t) - u_t(X_t \mid X_1) \rVert^2,$$
where $t \sim \mathrm{Unif}[0,1]$, $X_0 \sim p$ and $X_1 \sim q$. By inserting your result from Question 9, you get the final form which can be directly implemented in code.

The first code block defines the `Flow` as a neural network. It has three hidden layers with a default hidden dimension of 64 neurons. As input, it receives the time $t \in [0,1]$, along with the two coordinates of $x$. The `step` function performs a single [midpoint step](https://en.wikipedia.org/wiki/Midpoint_method) for solving the ODE.

In [ ]:
from torch import Tensor

import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

class Flow(nn.Module):
    def __init__(self, dim: int = 2, h: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, h), nn.ELU(),
            nn.Linear(h, h), nn.ELU(),
            nn.Linear(h, h), nn.ELU(),
            nn.Linear(h, dim))
    
    def forward(self, t: Tensor, x_t: Tensor) -> Tensor:
        return self.net(torch.cat((t, x_t), -1))
    
    def step(self, x_t: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor:
        t_start = t_start.view(1, 1).expand(x_t.shape[0], 1)

        u = self(t_start + (t_end - t_start) / 2, x_t + self(x_t, t_start) * (t_end - t_start) / 2)
        return x_t + (t_end - t_start) * u

In the following block, the velocity field is trained.

#### Question 10
Complete the code block below by entering the target velocity.

In [ ]:
flow = Flow()

optimizer = torch.optim.Adam(flow.parameters(), 1e-2)
loss_fn = nn.MSELoss()

for _ in range(10000):
    x_1 = Tensor(make_moons(256, noise=0.05)[0])
    x_0 = torch.randn_like(x_1)
    t = torch.rand(len(x_1), 1)
    
    x_t = (1 - t) * x_0 + t * x_1
    target = ...  # TODO: Target velocity
    
    optimizer.zero_grad()
    loss_fn(flow(t, x_t), target).backward()
    optimizer.step()

Finally, the resulting samples are plotted. Feel free to play around with the number of steps `n_steps` to see how it affects the quality of the samples.

In [ ]:
x = torch.randn(300, 2)
n_steps = 8
fig, axes = plt.subplots(1, n_steps + 1, figsize=(30, 4), sharex=True, sharey=True)
time_steps = torch.linspace(0, 1.0, n_steps + 1)

axes[0].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10)
axes[0].set_title(f't = {time_steps[0]:.2f}')
axes[0].set_xlim(-3.0, 3.0)
axes[0].set_ylim(-3.0, 3.0)

for i in range(n_steps):
    x = flow.step(x_t=x, t_start=time_steps[i], t_end=time_steps[i + 1])
    axes[i + 1].scatter(x.detach()[:, 0], x.detach()[:, 1], s=10)
    axes[i + 1].set_title(f't = {time_steps[i + 1]:.2f}')

plt.tight_layout()
plt.show()

While we only did it for a 2d dataset, the underlying principle is the _exact same_ for higher-dimensional data, such as images or videos. To make the right mental connection, think of each dot in the above plot as an entire image.

## References
[1] Diederik Kingma and Max Welling: [Auto-Encoding Variational Bayes](https://arxiv.org/pdf/1312.6114) (ICLR 2014) <br>
[2] Yaron Lipman et al.: [Flow Matching Guide and Code](https://arxiv.org/pdf/2412.06264)